# Cluster-Based Relabeling for All Participants
Pipeline:
1. Load each participant's CSV
2. Map `Annotator_1` → numeric drowsiness level
3. Remove inconsistent labels (KNN vote-based)
4. PCA + KMeans clustering
5. Relabel each point to its cluster's majority label
6. Save to `Clustered_Participants_Data/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mode
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [ ]:
BASE_DIR = "/home/karthik/Desktop/llm_eval/Refined_Participants_Data"
OUTPUT_DIR = "/home/karthik/Desktop/llm_eval/Clustered_Participants_Data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PARTICIPANTS = [
    {"id": "01_V",  "folder": "01_V_Data",  "file": "V_Data.csv"},
    {"id": "02_MK", "folder": "02_MK_Data", "file": "MK_Data.csv"},
    {"id": "03_A",  "folder": "03_A_Data",  "file": "A_Data.csv"},
    {"id": "04_SB", "folder": "04_SB_Data", "file": "SB_Data.csv"},
    {"id": "05_TG", "folder": "05_TG_Data", "file": "TG_Data.csv"},
    {"id": "06_MV", "folder": "06_MV_Data", "file": "MV_Data.csv"},
    {"id": "07_GK", "folder": "07_GK_Data", "file": "GK_Data.csv"},
    {"id": "08_D",  "folder": "08_D_Data",  "file": "D_Data.csv"},
]

FEATURES = [
    'window_id', 'metric_PERCLOS', 'metric_BlinkRate',
    'blink_duration_mean', 'blink_duration_std', 'blink_duration_max',
    'metric_YawnRate', 'metric_Entropy', 'metric_SteeringRate',
    'metric_SDLP', 'HRV_SDNN', 'HRV_RMSSD', 'HRV_SD1',
    'HRV_HF', 'HRV_WaveletEntropy', 'HRV_LFHF'
]

LABEL_MAP = {'Low': 1, 'Moderate': 2, 'High': 3}

In [ ]:
def cluster_relabel(df, features, n_clusters=3):
    """PCA (2D) → KMeans → relabel each point to its cluster's majority label."""
    X = StandardScaler().fit_transform(df[features])
    pcs = PCA(n_components=2).fit_transform(X)
    clusters = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(pcs)
    y = df['drowsiness_level'].values.copy()
    new_labels = y.copy()
    high_label = LABEL_MAP['High']   # protect High (3) from being relabeled down
    for c in np.unique(clusters):
        idx = np.where(clusters == c)[0]
        majority = mode(y[idx], keepdims=True).mode[0]
        for i in idx:
            if y[i] != majority and y[i] != high_label:
                new_labels[i] = majority
    return new_labels

In [ ]:
summary = []

for p in PARTICIPANTS:
    csv_path = os.path.join(BASE_DIR, p["folder"], p["file"])
    print(f"\n{'='*60}")
    print(f"Participant: {p['id']}")

    # ── 1. Load ──────────────────────────────────────────────────
    df = pd.read_csv(csv_path)

    # ── 2. Map labels — drop only rows with no annotation at all ─
    df['drowsiness_level'] = df['Annotator_1'].map(LABEL_MAP)
    df = df.dropna(subset=['drowsiness_level'])
    df['drowsiness_level'] = df['drowsiness_level'].astype(int)
    orig_labels = df['drowsiness_level'].value_counts().sort_index().to_dict()
    print(f"  Rows: {len(df)}, Original labels: {orig_labels}")

    # ── 3. Select features + impute NaN with column mean ─────────
    avail_features = [f for f in FEATURES if f in df.columns]
    df = df[avail_features + ['drowsiness_level']].copy()
    nan_count = df[avail_features].isna().sum().sum()
    df[avail_features] = df[avail_features].fillna(df[avail_features].mean())
    print(f"  NaN cells filled with mean: {nan_count}")

    # ── 4. Cluster & relabel ─────────────────────────────────────
    n_clusters = max(2, min(3, df['drowsiness_level'].nunique()))
    new_labels = cluster_relabel(df, avail_features, n_clusters=n_clusters)
    df['drowsiness_level'] = new_labels
    final_labels = df['drowsiness_level'].value_counts().sort_index().to_dict()
    print(f"  Final labels: {final_labels}")

    # ── 5. Save ──────────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_DIR, f"{p['id']}_clustered.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved → {out_path}")

    summary.append({
        'participant': p['id'],
        'rows': len(df),
        'original_labels': orig_labels,
        'final_labels': final_labels,
    })

print("\n" + "="*60)
print("All participants processed.")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────
rows = []
for s in summary:
    rows.append({
        'Participant': s['participant'],
        'Rows': s['rows_after_cleaning'],
        'Low (1) orig':  s['original_labels'].get(1, 0),
        'Mod (2) orig':  s['original_labels'].get(2, 0),
        'High (3) orig': s['original_labels'].get(3, 0),
        'Low (1) new':   s['final_labels'].get(1, 0),
        'Mod (2) new':   s['final_labels'].get(2, 0),
        'High (3) new':  s['final_labels'].get(3, 0),
    })
pd.DataFrame(rows).set_index('Participant')

In [ ]:
# ── Per-participant PCA scatter plots (re-run clustering for visualisation) ──
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax, p in zip(axes, PARTICIPANTS):
    out_path = os.path.join(OUTPUT_DIR, f"{p['id']}_clustered.csv")
    df = pd.read_csv(out_path)
    avail_features = [f for f in FEATURES if f in df.columns]

    X = StandardScaler().fit_transform(df[avail_features])
    pcs = PCA(n_components=2).fit_transform(X)
    y = df['drowsiness_level'].values

    sc = ax.scatter(pcs[:, 0], pcs[:, 1], c=y, cmap='viridis',
                    vmin=1, vmax=3, s=30, alpha=0.8)
    ax.set_title(p['id'])
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig.colorbar(sc, ax=axes[-1], label='Drowsiness (1=Low, 2=Mod, 3=High)')
plt.suptitle('PCA — Clustered & Relabeled Labels per Participant', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'all_participants_pca.png'), dpi=120)
plt.show()

In [ ]:
# ── Confirm saved files ───────────────────────────────────────────
print("Files in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(full)
    print(f"  {f:40s}  {size:>8,} bytes")